# Class 0: Data Setup & Preparation

## GIS Vulnerability & Risk Assessment: Climate Flooding in Buncombe County, NC

Welcome to **MSER 510: GIS Vulnerability & Risk Assessment**!

In this course, we will build a complete **Climate Vulnerability and Risk Assessment for flooding in Buncombe County, North Carolina** — a real-world analysis that community planners and emergency managers actually use.

### What is This Course?

This 9-notebook course teaches you how geographic information system (GIS) professionals assess climate risks. We will:

1. **Class 0 (this notebook)**: Set up our data and study area
2. **Class 1**: Analyze historical flood patterns
3. **Class 2**: Understand elevation and flood depth
4. **Class 3**: Map vulnerable populations
5. **Class 4**: Analyze critical infrastructure at risk
6. **Class 5**: Calculate exposure metrics
7. **Class 6**: Assess compound and cascading risks
8. **Class 7**: Model future climate scenarios
9. **Class 8**: Create maps and recommendations for decision-makers

**By the end of Class 8, you will have created professional maps and analysis that could be presented to a city council or county commission.**

### Why Buncombe County?

Buncombe County (Asheville, NC) sits along the French Broad River and Swannanoa River. These rivers flood regularly — sometimes severely. The county is also growing rapidly, with new developments in flood-prone areas. This makes it a perfect case study for vulnerability assessment.

### What Tools Do We Use?

We use **Python** and **Google Colab** (a free web-based notebook tool). No ArcGIS, no QGIS, no programming experience required.

Think of Python like a special calculator that can work with maps and data. We'll teach you the exact commands you need.

### What is a GeoPackage?

A **GeoPackage** (`.gpkg` file) is like a super-smart Excel spreadsheet that understands geography.

- Regular Excel: stores rows and columns of data
- GeoPackage: stores rows and columns of data, PLUS map shapes (boundaries, lines, points)

For example, a GeoPackage might have a "parcels" table with:
- Column 1: Parcel ID
- Column 2: Owner name
- Column 3: Land value
- Column 4: **The actual shape** (polygon) of where that parcel is on the map

One GeoPackage file can hold multiple tables (called "layers"), all perfectly aligned geographically.

### Why Google Colab?

**Google Colab** is a free web-based Python notebook. Think of it like Google Docs, but for code instead of documents.

Advantages:
- **Free** — no software to install on your computer
- **Runs in the browser** — Windows, Mac, Linux all work the same
- **Connects to Google Drive** — save all your data and notebooks there
- **Powerful computers** — Colab gives you free computing power (perfect for geospatial analysis)
- **No installation headaches** — all libraries are pre-installed or easy to add

Disadvantages:
- **Internet required** — you need to be online
- **Sessions time out** — if you leave it running for 12 hours, it stops (but your data in Google Drive is safe)
- **Less control** — you can't install specialized software like QGIS

For learning GIS analysis, Colab is perfect.

---

## Learning Objectives for Class 0

By the end of this notebook, you will:

1. ✓ Understand what libraries are and how to install them
2. ✓ Connect Python to Google Drive (to save files permanently)
3. ✓ Define your study area (Swannanoa watershed in Buncombe County)
4. ✓ Download real parcel data from North Carolina's database
5. ✓ Download FEMA flood zone maps
6. ✓ Download building footprints from OpenStreetMap data
7. ✓ Organize all this data into a single GeoPackage file
8. ✓ Create maps to verify the data looks correct
9. ✓ Know how to do all this in QGIS or ArcGIS Pro if you want to

---

## Step 1: Install and Import Libraries

Think of **libraries** like apps on your phone.
- When you first buy a phone, you install apps (like maps, calculator, email)
- Then you open the apps when you need them

In Python:
- **Installing** a library = downloading it and putting it on your computer (or in Colab's computers)
- **Importing** a library = opening the app and telling Python you want to use it

Below, we will install geographic libraries. Each one gives Python a new superpower:

### The Libraries We Need

| Library | What It Does | Analogy |
|---------|---------|----------|
| **geopandas** | Works with map data (shapes, points, boundaries) | Google Maps for Python |
| **fiona** | Reads and writes map files | File opener for geographic data |
| **shapely** | Does geometry math (overlaps, distances, intersections) | A calculator for shapes |
| **pyproj** | Handles coordinate systems & map projections | GPS translator (makes maps line up correctly) |
| **requests** | Downloads files from the internet | A web browser for code |
| **duckdb** | Fast queries on huge datasets | A super-fast database |
| **folium** | Creates interactive web maps | Mini Google Maps in your notebook |
| **matplotlib** | Creates static charts and maps | Python's drawing tool |
| **seaborn** | Makes statistical charts look beautiful | Makes matplotlib prettier |

**Key Concept: Geographic Coordinate Systems**

Maps need to know what coordinate system they're using. There are many:
- **EPSG:4326** = Global (latitude/longitude in degrees) — used by Google Maps
- **EPSG:6543** = North Carolina State Plane — more accurate for NC measurements in feet

We'll mostly use EPSG:4326 for downloads, then convert to EPSG:6543 (or EPSG:2264) for analysis.

Let's install these libraries now:

In [ ]:
# INSTALLATION: Download and set up all libraries
# This runs only once (on your first use)

!pip install geopandas fiona shapely pyproj requests duckdb folium seaborn --quiet

print("Libraries installed successfully!")

### Now Let's Import These Libraries

**Importing** means telling Python "I want to use this library in my code." We do this with the `import` statement.

When you import a library, you usually give it a short nickname:
- `import geopandas as gpd` means "use geopandas, but call it gpd in this code"
- This saves typing — `gpd.read_file()` is shorter than `geopandas.read_file()`


In [ ]:
# IMPORTS: Load all the libraries we need

import geopandas as gpd
import fiona
from shapely.geometry import box, Point, Polygon
import pyproj
import pandas as pd
import numpy as np
import requests
import json
import os
import shutil
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import folium
import seaborn as sns

try:
    import duckdb
    DUCKDB_AVAILABLE = True
except ImportError:
    DUCKDB_AVAILABLE = False
    print("Note: DuckDB not available in this environment")

print("All libraries imported successfully!")
print(f"DuckDB available: {DUCKDB_AVAILABLE}")

## Step 2: Connect to Google Drive

Google Colab runs on Google's computers in the cloud. To save files permanently, we need to connect to your Google Drive.

When you run the cell below, Colab will ask for permission to access your Google Drive. This is normal.

After that, any files we save will go to your Google Drive and be there forever.

### Folder Structure

We'll create this structure in your Google Drive:
```
MyDrive/
└── MSER_510_VULNERABILTY/
    ├── data/           (downloaded raw data)
    └── outputs/        (processed files, maps, analysis)
```


In [ ]:
# GOOGLE DRIVE: Connect to your Google Drive

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive connected!")

### Set Up Folder Structure

In [ ]:
# CREATE FOLDERS: Set up our data structure

BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Folder structure created!")
print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

### Helper Function: Query ArcGIS REST Services

Many government agencies share geographic data through **ArcGIS REST APIs**. These are web services that let you download data by asking for it over the internet.

The function below handles pagination (downloads data in chunks), converts to GeoDataFrame, and handles errors gracefully.


In [ ]:
def query_arcgis_rest(url, where='1=1', geometry=None, geometry_type='esriGeometryEnvelope',
                      out_fields='*', spatial_rel='esriSpatialRelIntersects',
                      out_sr='4326', in_sr='4326', return_geometry=True, result_offset=0,
                      result_record_count=2000):
    '''Download features from ArcGIS REST service with pagination support.

    This function handles:
    - Pagination (downloads data in chunks if there are more records than the limit)
    - Spatial filtering (only download features within a bounding box)
    - Coordinate system specification (tells the server what CRS the bounding box uses)
    - Error handling (prints helpful messages if something goes wrong)
    '''
    all_features = []

    while True:
        params = {
            'where': where,
            'outFields': out_fields,
            'outSR': out_sr,
            'f': 'geojson',
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': result_offset,
            'resultRecordCount': result_record_count,
        }

        # If a bounding box or geometry was provided, add it to the query
        if geometry:
            # ArcGIS REST services need to know what coordinate system the
            # bounding box is in. We embed the spatialReference in the geometry
            # AND pass inSR as a separate parameter. Some servers check one,
            # some check the other, so we send both to be safe.
            if isinstance(geometry, dict) and 'xmin' in geometry:
                # It's an envelope/bounding box — add spatialReference inside it
                geom_with_sr = geometry.copy()
                geom_with_sr['spatialReference'] = {'wkid': int(in_sr)}
                params['geometry'] = json.dumps(geom_with_sr)
            else:
                params['geometry'] = json.dumps(geometry) if isinstance(geometry, dict) else geometry

            params['geometryType'] = geometry_type
            params['spatialRel'] = spatial_rel
            params['inSR'] = in_sr  # Coordinate system of our bounding box

        try:
            response = requests.get(f"{url}/query", params=params, timeout=60)
            response.raise_for_status()
            data = response.json()

            if 'error' in data:
                print(f"  Error from server: {data['error']}")
                # If we get a 400 error with geometry, try without spatial filter
                if data['error'].get('code') == 400 and geometry:
                    print("  Tip: The server may not accept this geometry format.")
                    print("  Trying with WHERE clause only (no spatial filter)...")
                    params.pop('geometry', None)
                    params.pop('geometryType', None)
                    params.pop('spatialRel', None)
                    params.pop('inSR', None)
                    response = requests.get(f"{url}/query", params=params, timeout=60)
                    data = response.json()
                    if 'error' in data:
                        print(f"  Still getting error: {data['error']}")
                        break
                    features = data.get('features', [])
                    if features:
                        all_features.extend(features)
                        print(f"  Success with WHERE clause only! Got {len(features)} features.")
                    break
                break

            features = data.get('features', [])

            if not features:
                break

            all_features.extend(features)

            if len(features) < result_record_count:
                break

            result_offset += result_record_count
            print(f"    Downloaded {len(all_features)} features so far...")

        except requests.exceptions.RequestException as e:
            print(f"  Network error: {e}")
            print("  Check your internet connection and try running this cell again.")
            break
        except json.JSONDecodeError as e:
            print(f"  Error reading server response: {e}")
            break

    if all_features:
        geojson = {'type': 'FeatureCollection', 'features': all_features}
        try:
            gdf = gpd.GeoDataFrame.from_features(geojson, crs=f'EPSG:{out_sr}')
            print(f"  Downloaded {len(gdf)} total features")
            return gdf
        except Exception as e:
            print(f"  Error converting to GeoDataFrame: {e}")
            return gpd.GeoDataFrame()
    else:
        print(f"  No features found")
        return gpd.GeoDataFrame()

print("ArcGIS REST query function defined!")

## Step 3: Define the Study Area

We need to define the geographic area we're analyzing. We'll use the **Swannanoa watershed area** in Buncombe County, NC.

### What is a Watershed?

A **watershed** is the area of land where all water flows downhill into a specific river or stream.

For example:
- If you live on a hillside in the Swannanoa watershed, rain falling on your roof eventually flows into the Swannanoa River
- The Swannanoa River joins the French Broad River
- The French Broad River flows into the Tennessee River

Watersheds are natural boundaries for climate and flood analysis because they define how water moves across the landscape.

### Swannanoa Watershed Boundaries

The Swannanoa is a major sub-watershed in Buncombe County, NC. We'll use approximate coordinates:

- **West boundary**: -82.53° longitude (East Asheville / Tunnel Rd area)
- **East boundary**: -82.32° longitude (Black Mountain)
- **South boundary**: 35.56° latitude
- **North boundary**: 35.62° latitude

In [ ]:
# STUDY AREA: Define the Swannanoa watershed bounding box

swannanoa_west = -82.53
swannanoa_east = -82.32
swannanoa_south = 35.56
swannanoa_north = 35.62

# Create the bounding box as a dictionary (GeoJSON format)
study_area_bbox = {
    'xmin': swannanoa_west,
    'ymin': swannanoa_south,
    'xmax': swannanoa_east,
    'ymax': swannanoa_north
}

# Create a polygon (the actual shape we'll use for clipping)
study_area_polygon = box(swannanoa_west, swannanoa_south,
                         swannanoa_east, swannanoa_north)

# Convert to a GeoDataFrame
study_area_gdf = gpd.GeoDataFrame(
    {'name': ['Swannanoa Watershed Area'], 'id': [1]},
    geometry=[study_area_polygon],
    crs='EPSG:4326'
)

print("Study area defined!")
print(f"Swannanoa Watershed Bounding Box:")
print(f"  West:  {swannanoa_west}°")
print(f"  East:  {swannanoa_east}°")
print(f"  South: {swannanoa_south}°")
print(f"  North: {swannanoa_north}°")
print(f"\nArea as GeoDataFrame:")
print(study_area_gdf)

### Visualize and Adjust Your Study Area

The map below shows your study area bounding box on an interactive basemap. You can:

- **Zoom and pan** to explore the area
- **See the default Swannanoa boundary** drawn as a blue rectangle
- **See coordinates** as you move your mouse (shown in the bottom-left corner)
- **Click anywhere** on the map to see exact lat/lon coordinates in a popup

**Want to draw your own study area?** Use the drawing tools on the left side of the map:
1. Click the **rectangle** tool (square icon)
2. Draw a rectangle on the map
3. **Click the corners** of your drawn rectangle to read the coordinates from the popups
4. Update the coordinate values in the **next code cell** below the map
5. Re-run the cells from that point onward

This is a great way to explore Buncombe County before committing to a study area.

In [ ]:
# INTERACTIVE MAP: Visualize the study area on a basemap
# - Mouse position shown in bottom-left as you move the cursor
# - Click anywhere to see exact coordinates in a popup
# - Use the rectangle draw tool to sketch a new study area

import folium
from folium.plugins import Draw, MousePosition
from IPython.display import HTML, display

# Center the map on our study area
center_lat = (swannanoa_south + swannanoa_north) / 2
center_lon = (swannanoa_west + swannanoa_east) / 2

# Create the map with CartoDB Dark Matter basemap
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
    attr='CartoDB',
    width='100%',
    height='450px'
)

# Draw the current bounding box as a cyan rectangle (visible on dark basemap)
bbox_coords = [
    [swannanoa_south, swannanoa_west],
    [swannanoa_north, swannanoa_west],
    [swannanoa_north, swannanoa_east],
    [swannanoa_south, swannanoa_east],
    [swannanoa_south, swannanoa_west],
]

folium.Polygon(
    locations=bbox_coords,
    color='#00e5ff',
    weight=3,
    fill=True,
    fill_color='#00e5ff',
    fill_opacity=0.08,
    popup=f'Current Study Area\nWest: {swannanoa_west}\nEast: {swannanoa_east}\nSouth: {swannanoa_south}\nNorth: {swannanoa_north}',
    tooltip='Current Study Area (click for coordinates)'
).add_to(m)

# Add corner markers with coordinate labels
for label, lat, lon in [
    ('SW', swannanoa_south, swannanoa_west),
    ('NW', swannanoa_north, swannanoa_west),
    ('NE', swannanoa_north, swannanoa_east),
    ('SE', swannanoa_south, swannanoa_east),
]:
    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color='#00e5ff',
        fill=True,
        fill_color='white',
        fill_opacity=1,
        popup=f'{label}: lat={lat}, lon={lon}',
        tooltip=f'{label}: {lat}, {lon}'
    ).add_to(m)

# Show mouse position coordinates in the bottom-left corner as cursor moves
MousePosition(
    position='bottomleft',
    separator=' | ',
    prefix='Coordinates:',
    lat_formatter="function(lat){return 'Lat: ' + lat.toFixed(5)}",
    lng_formatter="function(lng){return 'Lon: ' + lng.toFixed(5)}",
    num_digits=5
).add_to(m)

# Click anywhere to see a popup with exact coordinates
folium.LatLngPopup().add_to(m)

# Drawing tools — rectangle only
Draw(
    draw_options={
        'rectangle': {'shapeOptions': {'color': '#ff5722', 'weight': 2}},
        'polygon': False,
        'circle': False,
        'circlemarker': False,
        'marker': False,
        'polyline': False,
    },
    edit_options={'edit': False}
).add_to(m)

print("Interactive Study Area Map")
print("=" * 50)
print(f"  Cyan rectangle = current study area")
print(f"  Mouse coordinates shown in bottom-left corner")
print(f"  Click anywhere on the map for exact lat/lon")
print(f"  Draw a rectangle to plan a new study area")

map_html = m._repr_html_()
display(HTML(f'<div style="height:450px;overflow:hidden;">{map_html}</div>'))

### Update Your Study Area Coordinates

If you drew a new rectangle on the map above, update the coordinates below.

**How to get coordinates from the map:**
1. Click the **southwest corner** (bottom-left) of your drawn rectangle — a popup shows the lat/lon
2. Click the **northeast corner** (top-right) — another popup shows those coordinates
3. Update the four values below with what you read from the popups
4. Run this cell, then continue with the rest of the notebook

If you're happy with the default Swannanoa corridor area, **just skip this cell** — no changes needed.

In [ ]:
# ============================================================
# UPDATE STUDY AREA COORDINATES HERE (if you drew a new box)
# ============================================================
# Replace these values with coordinates from the map above.
# Click corners of your drawn rectangle to read the lat/lon.
#
# Default values = Swannanoa corridor (East Asheville to Black Mountain)
# If you did NOT draw a new area, just skip this cell.
# ============================================================

swannanoa_west = -82.53
swannanoa_east = -82.32
swannanoa_south = 35.56
swannanoa_north = 35.62

# ============================================================
# DO NOT EDIT BELOW THIS LINE — this rebuilds the study area
# ============================================================

study_area_bbox = {
    'xmin': swannanoa_west,
    'ymin': swannanoa_south,
    'xmax': swannanoa_east,
    'ymax': swannanoa_north
}

study_area_polygon = box(swannanoa_west, swannanoa_south,
                         swannanoa_east, swannanoa_north)

study_area_gdf = gpd.GeoDataFrame(
    {'name': ['Swannanoa Watershed Area'], 'id': [1]},
    geometry=[study_area_polygon],
    crs='EPSG:4326'
)

print('Study area updated!')
print(f'  West:  {swannanoa_west}')
print(f'  East:  {swannanoa_east}')
print(f'  South: {swannanoa_south}')
print(f'  North: {swannanoa_north}')

## Step 4: Download Parcel Data from NC OneMap

**Parcels** are the individual property boundaries that appear on property tax maps.

Each parcel record includes:
- **Parcel ID**: Unique identifier
- **PARUSECODE**: Numeric code for land use
- **PARUSEDESC**: Text description of land use
- **STRUCTYEAR**: Year the main building was constructed
- **Geometry**: The actual polygon shape showing where the parcel is

### Why Parcels Matter for Risk Assessment

When assessing flooding vulnerability, we need to know:
- What's on the land? (house, factory, farm, empty lot?)
- What's it worth? (assessed value)
- How old is it? (older buildings may be less resilient)

### The NC OneMap Service

North Carolina maintains the **NC OneMap** service that provides free parcel data:
```
https://services.nconemap.gov/secure/rest/services/NC1Map_Parcels/MapServer/1
```

We'll query this service for all parcels in our study area.


In [ ]:
# PARCELS: Download from NC OneMap
#
# The NC OneMap parcel service uses LOWERCASE field names.
# The county FIPS field is "cntyfips" (not "COUNTYFIPS").
# Buncombe County FIPS code is "021".
# The service's native coordinate system is EPSG:2264 (NC State Plane, feet).

print("\nDownloading Parcels from NC OneMap...")
print("(This may take a few minutes — the service returns up to 5000 records per page)\n")

nc_parcels_url = "https://services.nconemap.gov/secure/rest/services/NC1Map_Parcels/MapServer/1"

# The field name on this service is lowercase "cntyfips"
# Buncombe County FIPS = '021'
where_clause = "cntyfips = '021'"

print(f"Filter: {where_clause}")
print(f"Bounding box (lat/lon): {study_area_bbox}")

# Convert our lat/lon bounding box to the service's native coordinate system
# (EPSG:2264 = NC State Plane in US feet). The NC OneMap server works best
# when the spatial filter uses the same coordinates it stores internally.
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:4326", "EPSG:2264", always_xy=True)

# Transform the corners of our bounding box
x_min, y_min = transformer.transform(study_area_bbox['xmin'], study_area_bbox['ymin'])
x_max, y_max = transformer.transform(study_area_bbox['xmax'], study_area_bbox['ymax'])

bbox_native = {
    'xmin': x_min, 'ymin': y_min,
    'xmax': x_max, 'ymax': y_max
}
print(f"Bounding box (NC State Plane ft): xmin={x_min:.0f}, ymin={y_min:.0f}, xmax={x_max:.0f}, ymax={y_max:.0f}")

# Download parcels using native coordinates for the spatial filter
parcels_raw = query_arcgis_rest(
    url=nc_parcels_url,
    where=where_clause,
    geometry=bbox_native,
    in_sr='2264',        # Bounding box is in NC State Plane feet
    out_sr='4326',       # We want results back in lat/lon
    result_record_count=5000  # Service max is 5000
)

print(f"\nDownloaded {len(parcels_raw)} parcel records")

if len(parcels_raw) > 0:
    print(f"\nParcel columns available:")
    for col in sorted(parcels_raw.columns):
        print(f"  - {col}")

    # NC OneMap field names (all lowercase):
    #   improvval   = improvement/building value
    #   landval     = land value
    #   parval      = total parcel value
    #   structyear  = year the structure was built
    #   parusecode  = land use code
    #   parusedesc  = land use description
    value_cols = [c for c in parcels_raw.columns if any(v in c.lower() for v in
                  ['val', 'value', 'improv', 'landval', 'parval'])]
    if value_cols:
        print(f"\nValue-related columns found: {value_cols}")

    print(f"\nFirst 3 parcels:")
    print(parcels_raw.head(3))
else:
    print("Warning: No parcels downloaded.")
    print("Possible causes:")
    print("  - The NC OneMap server may be temporarily unavailable")
    print("  - Try running this cell again in a few minutes")
    print("\nTroubleshooting: Test the service manually at:")
    print("  https://services.nconemap.gov/secure/rest/services/NC1Map_Parcels/MapServer/1/query")
    print("  Set Where to: cntyfips = '021'  |  Out Fields: *  |  Format: GeoJSON")

### Clip Parcels to Study Area

The bounding box we used is rectangular. We should **clip** the parcels to keep only those that intersect with our study area polygon.

**Clipping** is a spatial operation: "Keep only the features that intersect with this boundary."


In [ ]:
# CLIP: Keep only parcels that intersect the study area polygon

if len(parcels_raw) > 0:
    print(f"Before clipping: {len(parcels_raw)} parcels")

    # Clip to study area boundary
    parcels_clipped = gpd.clip(parcels_raw, study_area_gdf)
    print(f"After clipping to study area: {len(parcels_clipped)} parcels")

    # Keep ALL columns exactly as they come from NC OneMap (no renaming)
    parcels = parcels_clipped.copy()

    # IMPORTANT: Fill null/empty values so NO parcels are excluded from summaries
    if 'parusedesc' in parcels.columns:
        parcels['parusedesc'] = parcels['parusedesc'].fillna('Unknown')
        parcels.loc[parcels['parusedesc'].str.strip() == '', 'parusedesc'] = 'Unknown'
    if 'parusecode' in parcels.columns:
        parcels['parusecode'] = parcels['parusecode'].fillna('Unknown')
        parcels.loc[parcels['parusecode'].astype(str).str.strip() == '', 'parusecode'] = 'Unknown'

    # Print the key fields we'll use in later classes
    print("\nKey fields for later classes (all lowercase from NC OneMap):")
    key_fields = {
        'parusecode':  'Land use code (e.g. residential, commercial)',
        'parusedesc':  'Land use description',
        'parusecd2':   'Secondary land use code',
        'parusedsc2':  'Secondary land use description',
        'structyear':  'Year the structure was built',
        'improvval':   'Improvement (building) value',
        'landval':     'Land value',
        'parval':      'Total parcel value',
        'parvaltype':  'Type of value reported',
        'parno':       'Parcel number (ID)',
        'cntyfips':    'County FIPS code',
    }
    for field, desc in key_fields.items():
        found = field in parcels.columns
        status = "FOUND" if found else "not found"
        print(f"  {field:15s} — {desc:45s} [{status}]")

    print(f"\nTotal columns: {len(parcels.columns)}")

    # ============================================================
    # SUMMARY TABLE: parusedesc and parusecode
    # ============================================================
    # This table shows EVERY parcel — nulls are labeled 'Unknown'
    # so the total always matches the clipped parcel count.
    # ============================================================
    print("\n" + "=" * 90)
    print(f"  PARCEL SUMMARY TABLE — {len(parcels):,} total parcels (none excluded)")
    print("=" * 90)

    # Build combined summary
    if 'parusedesc' in parcels.columns and 'parusecode' in parcels.columns:
        summary = parcels.groupby(['parusecode', 'parusedesc']).size().reset_index(name='Count')
        summary = summary.sort_values('parusecode')
        summary['Pct'] = (summary['Count'] / len(parcels) * 100).round(1)
        print(f"\n  {'parusecode':<15s} {'parusedesc':<40s} {'Count':>8s} {'Pct':>7s}")
        print(f"  {'-'*15} {'-'*40} {'-'*8} {'-'*7}")
        for _, row in summary.iterrows():
            print(f"  {str(row['parusecode']):<15s} {row['parusedesc']:<40s} {int(row['Count']):>8,} {row['Pct']:>6.1f}%")
        print(f"  {'-'*15} {'-'*40} {'-'*8} {'-'*7}")
        print(f"  {'TOTAL':<15s} {'':40s} {len(parcels):>8,} {'100.0':>6s}%")
    elif 'parusedesc' in parcels.columns:
        summary = parcels['parusedesc'].value_counts().reset_index()
        summary.columns = ['parusedesc', 'Count']
        summary['Pct'] = (summary['Count'] / len(parcels) * 100).round(1)
        print(f"\n  {'parusedesc':<40s} {'Count':>8s} {'Pct':>7s}")
        print(f"  {'-'*40} {'-'*8} {'-'*7}")
        for _, row in summary.iterrows():
            print(f"  {row['parusedesc']:<40s} {int(row['Count']):>8,} {row['Pct']:>6.1f}%")
        print(f"  {'-'*40} {'-'*8} {'-'*7}")
        print(f"  {'TOTAL':<40s} {len(parcels):>8,} {'100.0':>6s}%")
    else:
        print("  parusedesc field not found in data")

    print("\n" + "=" * 90)
    print("  Use the parusecode values above when choosing assets in Class 1.")
    print("=" * 90)

    # Save clipped parcels as backup
    parcels_gpkg_path = os.path.join(DATA_DIR, 'parcels_clipped.gpkg')
    parcels.to_file(parcels_gpkg_path, layer='parcels', driver='GPKG')
    print(f"\n  Saved clipped parcels to: {parcels_gpkg_path}")
else:
    parcels = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')
    print("No parcels to process")

### Map: Parcels by Land Use Description

Let's see what we downloaded! This map shows all the parcels in the study area, colored by their **parusedesc** (land use description) field from NC OneMap. This gives us a quick feel for the composition of the area — mostly residential? Commercial corridors? Open space?

In [ ]:
# MAP: Parcels colored by land use description (parusedesc) — interactive only

if len(parcels) > 0 and 'parusedesc' in parcels.columns:
    from IPython.display import HTML, display
    
    # Get the unique land use descriptions, sorted by frequency
    use_counts = parcels['parusedesc'].value_counts()
    unique_uses = use_counts.index.tolist()

    # Generate a bright color palette
    n_colors = len(unique_uses)
    if n_colors <= 10:
        palette = plt.cm.Set1(np.linspace(0, 1, min(n_colors, 9)))
    elif n_colors <= 20:
        palette = plt.cm.tab20(np.linspace(0, 1, 20))
    else:
        palette = plt.cm.gist_ncar(np.linspace(0.05, 0.95, n_colors))

    # Build a color dictionary: land use description -> hex color
    def rgba_to_hex(rgba):
        r, g, b = int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255)
        return f'#{r:02x}{g:02x}{b:02x}'

    use_colors = {}
    for i, use in enumerate(unique_uses):
        use_colors[use] = rgba_to_hex(palette[i % len(palette)])

    # Interactive folium map — dark basemap
    parcels_4326 = parcels.to_crs('EPSG:4326') if parcels.crs != 'EPSG:4326' else parcels
    center = [parcels_4326.geometry.centroid.y.mean(), parcels_4326.geometry.centroid.x.mean()]

    m2 = folium.Map(
        location=center, zoom_start=13,
        tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
        attr='CartoDB',
        width='100%', height='450px'
    )

    # Add ALL parcels to the map
    for _, row in parcels_4326.iterrows():
        use_desc = row.get('parusedesc', 'Unknown')
        color = use_colors.get(use_desc, '#999999')
        popup_text = f"<b>{use_desc}</b>"
        if 'parval' in row.index and pd.notna(row['parval']):
            popup_text += f"<br>Parcel Value: ${row['parval']:,.0f}"
        try:
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda x, c=color: {
                    'fillColor': c, 'color': '#555555', 'weight': 0.5, 'fillOpacity': 0.7
                },
                tooltip=use_desc,
                popup=folium.Popup(popup_text, max_width=200)
            ).add_to(m2)
        except Exception:
            pass

    print(f"Interactive map — {len(parcels):,} parcels")
    print("Click a parcel to see its land use description and value")
    map_html = m2._repr_html_()
    display(HTML(f'<div style="height:450px;overflow:hidden;">{map_html}</div>'))
else:
    print("No parcels to map (or parusedesc field not found)")

### Next Step: Choose Your Community Assets

Using the summary table above, identify the **parusecode** values you want to group into each community asset. Each asset is a list of one or more parusecode values.

For example, if you want "Residential" to include codes `100`, `101`, `102` and "Commercial" to include `200`, `201`, you would enter:

```python
community_asset_1 = ['100', '101', '102']
community_asset_2 = ['200', '201']
```

Edit the two lists in the next cell, then run it to save your selections. You can also update these in **Class 1** if you change your mind after reviewing additional summary tables.

In [ ]:
# ============================================================
# CHOOSE YOUR TWO COMMUNITY ASSET GROUPS HERE
# ============================================================
# Using the summary table above, enter the parusecode values
# for each community asset group as a Python list.
#
# Example — if you want "Residential" to include codes
# '100', '101', '102', and "Commercial" to include '200', '201':
#
#   community_asset_1 = ['100', '101', '102']
#   community_asset_2 = ['200', '201']
# ============================================================

community_asset_1 = ['100', '101', '105', '120', '121', '170', '173', '411', '416']
community_asset_2 = ['340', '341', '365', '405', '414', '415', '417', '421', '423',
                     '425', '426', '430', '431', '432', '434', '435', '438', '440',
                     '444', '446', '447', '448', '450', '454', '455', '456', '462',
                     '464', '466', '468', '470', '471', '472', '476', '477', '478',
                     '480', '481', '483', '490', '492', '494', '495', '512', '541',
                     '543', '544', '551', '554']

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

# Validate selections against actual data
if len(parcels) > 0 and 'parusecode' in parcels.columns:
    available = parcels['parusecode'].dropna().unique().tolist()

    for i, asset_codes in enumerate([community_asset_1, community_asset_2], 1):
        print(f"\n  Community Asset {i}: {asset_codes}")
        total = 0
        for code in asset_codes:
            code_str = str(code)
            if code_str in [str(a) for a in available]:
                count = (parcels['parusecode'].astype(str) == code_str).sum()
                # Look up the description for this code
                desc_match = parcels.loc[parcels['parusecode'].astype(str) == code_str, 'parusedesc']
                desc = desc_match.iloc[0] if len(desc_match) > 0 else '(no description)'
                print(f"    parusecode '{code_str}' — {desc} — {count:,} parcels")
                total += count
            else:
                print(f"    WARNING: parusecode '{code_str}' NOT FOUND in data!")
                print(f"             Check the summary table above for valid codes.")
        print(f"    TOTAL for Asset {i}: {total:,} parcels")

    # Load existing config (if any) or start fresh
    config_path = os.path.join(DATA_DIR, 'course_config.json')
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            course_config = json.load(f)
    else:
        course_config = {}

    # Save to config — store as lists of parusecode values
    course_config['community_asset_1'] = [str(c) for c in community_asset_1]
    course_config['community_asset_2'] = [str(c) for c in community_asset_2]
    course_config['study_area_bbox'] = study_area_bbox

    with open(config_path, 'w') as f:
        json.dump(course_config, f, indent=2)

    # Verify
    if os.path.exists(config_path):
        print(f"\n  Config saved to: {config_path}")
        with open(config_path, 'r') as f:
            verify = json.load(f)
        print(f"  Verified — Asset 1 codes: {verify['community_asset_1']}")
        print(f"  Verified — Asset 2 codes: {verify['community_asset_2']}")
        print(f"  Future notebooks will load your selections automatically.")
    else:
        print(f"\n  ERROR: Config file was NOT created at {config_path}")
else:
    print("No parcels loaded — run the download cells above first.")

## Step 5: Download Flood Zone Data from FEMA

**Flood zones** define areas at risk of flooding, mapped by the Federal Emergency Management Agency (FEMA).

FEMA's **National Flood Hazard Layer (NFHL)** provides flood maps for the entire United States. The key field is `FLD_ZONE`, which indicates the flood risk category:

- **AE, A, AH, AO, A99** — 100-year floodplain (1% annual chance of flooding)
- **X (shaded)** — 500-year floodplain (0.2% annual chance)
- **Floodway** — The channel where floodwater actually flows (most dangerous)

### The FEMA NFHL Service

We'll download flood zone polygons from FEMA's ArcGIS REST service:
```
https://hazards.fema.gov/gis/nfhl/rest/services/public/NFHL/MapServer/28
```

Layer 28 contains the **Flood Hazard Zones** polygons.

In [ ]:
# FLOOD ZONES: Download from FEMA National Flood Hazard Layer
#
# Primary: FEMA NFHL (Layer 28 = Flood Hazard Zones)
# Fallback: Esri Living Atlas (USA Flood Hazard Reduced Set)
# Both have the FLD_ZONE field we need.

import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("\nDownloading Flood Zones...")
print("(Trying FEMA first, then Esri Living Atlas as fallback)\n")

fema_url = "https://hazards.fema.gov/gis/nfhl/rest/services/public/NFHL/MapServer/28"
esri_url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_Flood_Hazard_Reduced_Set_gdb/FeatureServer/0"

geometry_param = json.dumps({
    "xmin": study_area_bbox['xmin'],
    "ymin": study_area_bbox['ymin'],
    "xmax": study_area_bbox['xmax'],
    "ymax": study_area_bbox['ymax'],
    "spatialReference": {"wkid": 4326}
})

def download_flood_zones(service_url, service_name, max_attempts=5):
    """Download flood zones from an ArcGIS REST service with retry logic."""
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))

    all_features = []
    offset = 0
    batch_size = 2000

    while True:
        params = {
            'where': '1=1',
            'outFields': '*',
            'outSR': '4326',
            'f': 'geojson',
            'returnGeometry': 'true',
            'resultOffset': offset,
            'resultRecordCount': batch_size,
            'geometry': geometry_param,
            'geometryType': 'esriGeometryEnvelope',
            'spatialRel': 'esriSpatialRelIntersects',
            'inSR': '4326'
        }

        success = False
        for attempt in range(1, max_attempts + 1):
            try:
                print(f"  [{service_name}] Batch at offset {offset} (attempt {attempt})...", end=' ')
                resp = session.get(f"{service_url}/query", params=params, timeout=60)
                resp.raise_for_status()
                data = resp.json()

                if 'features' in data and len(data['features']) > 0:
                    batch_gdf = gpd.GeoDataFrame.from_features(data['features'], crs='EPSG:4326')
                    all_features.append(batch_gdf)
                    print(f"got {len(batch_gdf)} features")
                    offset += len(batch_gdf)
                    success = True
                    if len(batch_gdf) < batch_size:
                        break
                else:
                    print("no more features")
                    success = True
                break

            except Exception as e:
                print(f"error: {type(e).__name__}")
                if attempt < max_attempts:
                    wait = 2 ** attempt
                    print(f"    Waiting {wait}s before retry...")
                    time.sleep(wait)

        if not success:
            session.close()
            return None
        if success and ('features' not in data or len(data.get('features', [])) < batch_size):
            break

    session.close()

    if all_features:
        result = pd.concat(all_features, ignore_index=True)
        return gpd.GeoDataFrame(result, geometry='geometry', crs='EPSG:4326')
    return None

# Try FEMA first
print(f"Source 1: FEMA NFHL")
print(f"  {fema_url}\n")
flood_zones_raw = download_flood_zones(fema_url, "FEMA")

# If FEMA failed, try Esri Living Atlas
if flood_zones_raw is None or len(flood_zones_raw) == 0:
    print(f"\n  FEMA download failed. Trying Esri Living Atlas fallback...")
    print(f"  {esri_url}\n")
    flood_zones_raw = download_flood_zones(esri_url, "Esri")

# Report results
if flood_zones_raw is not None and len(flood_zones_raw) > 0:
    print(f"\nTotal downloaded: {len(flood_zones_raw)} flood zone features")

    if 'FLD_ZONE' in flood_zones_raw.columns:
        print(f"\nFlood zones by FLD_ZONE:")
        print(flood_zones_raw['FLD_ZONE'].value_counts())
    else:
        print(f"\nAvailable columns: {list(flood_zones_raw.columns)}")

    # Save raw flood zones as backup
    flood_gpkg_path = os.path.join(DATA_DIR, 'flood_zones_raw.gpkg')
    flood_zones_raw.to_file(flood_gpkg_path, layer='flood_zones', driver='GPKG')
    print(f"\n  Saved raw flood zones to: {flood_gpkg_path}")


In [ ]:
# FALLBACK: Load flood zones from a previous download if FEMA is down
# Uncomment the lines below ONLY if the download cell above failed.

# flood_gpkg_path = os.path.join(DATA_DIR, 'flood_zones_raw.gpkg')
# if os.path.exists(flood_gpkg_path):
#     flood_zones_raw = gpd.read_file(flood_gpkg_path, layer='flood_zones')
#     print(f"Loaded {len(flood_zones_raw)} flood zones from: {flood_gpkg_path}")
# else:
#     print(f"No saved flood zones found at: {flood_gpkg_path}")
#     print("You need to run the download cell when FEMA is available.")

In [ ]:
# FLOOD CATEGORIZATION: Simplify zones into risk categories
#
# IMPORTANT: FLD_ZONE alone is not enough — floodways show as 'AE'
# just like regular 100-year zones. The ZONE_SUBTY field distinguishes
# floodways from regular flood zones.

if len(flood_zones_raw) > 0:
    print("Categorizing flood zones by risk level...")

    flood_zones = flood_zones_raw.copy()

    # Check which fields are available
    has_zone_subty = 'ZONE_SUBTY' in flood_zones.columns
    has_fld_zone = 'FLD_ZONE' in flood_zones.columns

    if has_zone_subty:
        print(f"  Using ZONE_SUBTY to identify floodways")
        print(f"\n  ZONE_SUBTY values:")
        print(flood_zones['ZONE_SUBTY'].value_counts().to_string())
    if has_fld_zone:
        print(f"\n  FLD_ZONE values:")
        print(flood_zones['FLD_ZONE'].value_counts().to_string())

    def categorize_flood_risk(row):
        fld_zone = row.get('FLD_ZONE', None)
        zone_subty = row.get('ZONE_SUBTY', None)

        # Check ZONE_SUBTY first — this is how floodways are identified
        if pd.notna(zone_subty):
            subty = str(zone_subty).strip().upper()
            if 'FLOODWAY' in subty:
                return 'Floodway'

        # Then check FLD_ZONE
        if pd.isna(fld_zone):
            return 'Unknown'

        zone = str(fld_zone).strip().upper()

        if 'FLOODWAY' in zone:
            return 'Floodway'
        elif zone in ['AE', 'A', 'AH', 'AO', 'A99']:
            return '100-year'
        elif zone.startswith('X'):
            return '500-year'
        else:
            return 'Other'

    flood_zones['flood_category'] = flood_zones.apply(categorize_flood_risk, axis=1)

    print(f"\nFinal flood categories:")
    print(flood_zones['flood_category'].value_counts().to_string())

    flood_zones = gpd.clip(flood_zones, study_area_gdf)
    print(f"\nAfter clipping: {len(flood_zones)} flood zones")

    keep_cols = ['FLD_ZONE', 'ZONE_SUBTY', 'flood_category', 'geometry']
    available = [c for c in keep_cols if c in flood_zones.columns]
    flood_zones = flood_zones[available].copy()

    print(f"\nSample flood zones:")
    print(flood_zones.head())

else:
    print("No flood zones to process")
    flood_zones = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

### Map: Flood Zones by Risk Category

Now let's see the flood zones. The most dangerous areas (Floodways — where the river actually flows during a flood) are shown in dark red. The 100-year floodplain is shown in orange, and the 500-year floodplain in yellow.

In [ ]:
# MAP: Flood zones colored by risk category — interactive only

if len(flood_zones) > 0:
    from IPython.display import HTML, display

    flood_colors = {
        'Floodway': '#2B5797',
        '100-year': '#8FABBE',
        '500-year': '#B4D4E7',
        'Other': '#B3E5FC',
        'Unknown': '#E0E0E0',
    }

    # Interactive flood zone map with dark basemap
    fz_4326 = flood_zones.to_crs('EPSG:4326') if flood_zones.crs != 'EPSG:4326' else flood_zones
    center = [fz_4326.geometry.centroid.y.mean(), fz_4326.geometry.centroid.x.mean()]

    m3 = folium.Map(
        location=center, zoom_start=12,
        tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
        attr='CartoDB',
        width='100%', height='450px'
    )

    for _, row in fz_4326.iterrows():
        cat = row.get('flood_category', 'Unknown')
        color = flood_colors.get(cat, '#E0E0E0')
        try:
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda x, c=color: {
                    'fillColor': c, 'color': '#555555', 'weight': 0.5, 'fillOpacity': 0.6
                },
                tooltip=cat
            ).add_to(m3)
        except Exception:
            pass

    # Print category counts
    print("Flood zones on map:")
    for cat in flood_colors:
        count = len(flood_zones[flood_zones['flood_category'] == cat])
        if count > 0:
            print(f"  {cat}: {count:,}")

    map_html = m3._repr_html_()
    display(HTML(f'<div style="height:450px;overflow:hidden;">{map_html}</div>'))
else:
    print("No flood zones to map")

## Step 5b: Download Building Footprints from Overture Maps

Building footprints tell us **where structures actually sit on the ground**. In Class 2, we need to know which parcels have buildings exposed to flooding — a parcel with a building on it has higher potential impact than vacant land.

### What is Overture Maps?

**Overture Maps** is a collaborative open-data project (backed by Amazon, Meta, Microsoft, and others) that provides free, high-quality map data including building footprints derived from satellite imagery and other sources.

We'll use **DuckDB** (already installed) to query Overture's cloud-hosted data directly — no account or API key needed. We filter to our study area bounding box so we only download what we need.

### Why Building Footprints Matter

- A flooded parcel with a building = **people and property at risk**
- A flooded parcel with no building = **lower immediate impact**
- Building footprints let us distinguish between these two cases in Class 2

In [ ]:
# BUILDINGS: Download building footprints from Overture Maps
# Uses DuckDB to query Overture's cloud-hosted parquet files
# filtered to our study area bounding box.

import duckdb

print("Downloading building footprints from Overture Maps...")
print(f"Study area: {swannanoa_west}, {swannanoa_south} to {swannanoa_east}, {swannanoa_north}")

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

# Query Overture buildings within our bounding box
# If this fails, check https://overturemaps.org for the current release date
# and update the date below.
query = f"""
SELECT
    id,
    names.primary AS name,
    class,
    ST_AsText(geometry) AS geometry_wkt
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-01-21.0/theme=buildings/type=building/*', hive_partitioning=1)
WHERE bbox.xmin BETWEEN {swannanoa_west} AND {swannanoa_east}
  AND bbox.ymin BETWEEN {swannanoa_south} AND {swannanoa_north}
  AND bbox.xmax BETWEEN {swannanoa_west} AND {swannanoa_east}
  AND bbox.ymax BETWEEN {swannanoa_south} AND {swannanoa_north}
"""

print("Querying Overture Maps (this may take 1-2 minutes)...")
result = con.execute(query).fetchdf()
con.close()

print(f"  Downloaded {len(result):,} building footprints")

# Convert to GeoDataFrame
if len(result) > 0:
    from shapely import wkt
    result['geometry'] = result['geometry_wkt'].apply(wkt.loads)
    buildings = gpd.GeoDataFrame(result.drop(columns=['geometry_wkt']), geometry='geometry', crs='EPSG:4326')

    # Clip to study area polygon (some buildings may extend beyond bbox)
    buildings = gpd.clip(buildings, study_area_gdf.to_crs('EPSG:4326'))
    print(f"  After clipping to study area: {len(buildings):,} buildings")

    if 'class' in buildings.columns:
        print(f"\n  Building classes:")
        for cls, count in buildings['class'].value_counts().head(10).items():
            print(f"    {cls}: {count:,}")
else:
    buildings = gpd.GeoDataFrame(columns=['id', 'name', 'class', 'geometry'], geometry='geometry', crs='EPSG:4326')
    print("  WARNING: No buildings found in study area")

print(f"\n✓ Building footprints ready ({len(buildings):,} features)")

## Step 6: Standardize Coordinate Systems

All our data needs to use the same coordinate system (CRS) so they line up properly.

**EPSG:4326** = Web standard (lat/lon)
**EPSG:6543** = North Carolina state plane (feet)

For this analysis, we'll convert everything to EPSG:6543 for accurate NC-specific measurements.


In [ ]:
# COORDINATE SYSTEMS: Standardize to North Carolina state plane

print("\nStandardizing coordinate systems...")

crs_web = 'EPSG:4326'
crs_nc = 'EPSG:6543'

print(f"Web CRS: {crs_web} (for downloads)")
print(f"NC CRS: {crs_nc} (for analysis)")

# Reproject all datasets
if len(study_area_gdf) > 0:
    study_area_gdf = study_area_gdf.to_crs(crs_nc)
    print(f"  study_area reprojected")

if len(parcels) > 0:
    parcels = parcels.to_crs(crs_nc)
    print(f"  parcels reprojected")

if len(flood_zones) > 0:
    flood_zones = flood_zones.to_crs(crs_nc)
    print(f"  flood_zones reprojected")

if len(buildings) > 0:
    buildings = buildings.to_crs(crs_nc)
    print(f"  buildings reprojected")

print(f"\nAll datasets now use NC state plane!")

## Step 7: Save All Data to a GeoPackage

Now we'll combine all our data into a single **GeoPackage** file. A GeoPackage can hold multiple layers:

1. `study_area` — our boundary polygon
2. `parcels` — property parcels with land use
3. `flood_zones` — FEMA flood hazard areas
4. `buildings` — building footprints from Overture Maps

Each layer will be a separate table in the file, all geographically aligned.

### Why GeoPackage?

- Single file (easy to share and backup)
- Multiple layers in one file (organized)
- Works with QGIS, ArcGIS Pro, and Python
- Open standard (not vendor-locked)

In [ ]:
# GEOPACKAGE: Create the GeoPackage file with all layers
#
# This cell saves ALL downloaded data into a single GeoPackage file
# in your Google Drive. Classes 1-8 will load this file.

print("\n" + "=" * 60)
print("SAVING DATA TO GEOPACKAGE")
print("=" * 60)

# This is the main file that ALL later notebooks (Classes 1-8) will load
gpkg_filename = 'vulnerability_risk_data.gpkg'

# Save to BASE_DIR root (where most classes look for it)
gpkg_filepath = os.path.join(DATA_DIR, gpkg_filename)

print(f"\nTarget file: {gpkg_filepath}")

# Remove existing file if present
if os.path.exists(gpkg_filepath):
    os.remove(gpkg_filepath)
    print("(Removed existing file to start fresh)")

layers_written = []
errors = []

# Layer 1: Study Area
print(f"\nWriting layers...")
try:
    study_area_gdf.to_file(gpkg_filepath, layer='study_area', driver='GPKG')
    layers_written.append('study_area')
    print(f"  [OK] study_area — {len(study_area_gdf)} feature(s)")
except Exception as e:
    errors.append(f"study_area: {e}")
    print(f"  [ERROR] study_area — {e}")

# Layer 2: Parcels
if len(parcels) > 0:
    try:
        parcels.to_file(gpkg_filepath, layer='parcels', driver='GPKG')
        layers_written.append('parcels')
        print(f"  [OK] parcels — {len(parcels):,} feature(s)")
    except Exception as e:
        errors.append(f"parcels: {e}")
        print(f"  [ERROR] parcels — {e}")
else:
    print(f"  [SKIP] parcels — no data downloaded")

# Layer 3: Flood Zones
if len(flood_zones) > 0:
    try:
        flood_zones.to_file(gpkg_filepath, layer='flood_zones', driver='GPKG')
        layers_written.append('flood_zones')
        print(f"  [OK] flood_zones — {len(flood_zones):,} feature(s)")
    except Exception as e:
        errors.append(f"flood_zones: {e}")
        print(f"  [ERROR] flood_zones — {e}")
else:
    print(f"  [SKIP] flood_zones — no data downloaded")

# Layer 4: Buildings (from Overture Maps)
if len(buildings) > 0:
    try:
        buildings.to_file(gpkg_filepath, layer='buildings', driver='GPKG')
        layers_written.append('buildings')
        print(f"  [OK] buildings — {len(buildings):,} feature(s)")
    except Exception as e:
        errors.append(f"buildings: {e}")
        print(f"  [ERROR] buildings — {e}")
else:
    print(f"  [SKIP] buildings — no data downloaded")

# Verify the file was created
print(f"\n--- Verification ---")
if os.path.exists(gpkg_filepath):
    file_size_mb = os.path.getsize(gpkg_filepath) / (1024 * 1024)
    print(f"  GeoPackage EXISTS: {gpkg_filepath}")
    print(f"  File size: {file_size_mb:.2f} MB")
    print(f"  Layers written: {', '.join(layers_written)}")

    # Verify by reading back the layer list
    try:
        verified_layers = fiona.listlayers(gpkg_filepath)
        print(f"  Verified layers in file: {', '.join(verified_layers)}")
    except Exception as e:
        print(f"  Could not verify layers: {e}")



    print(f"SUCCESS: GeoPackage saved with {len(layers_written)} layers")
    print(f"{'='*60}")
else:
    print(f"  ERROR: GeoPackage file was NOT created!")
    print(f"  Check the error messages above.")
    if errors:
        print(f"  Errors encountered:")
        for err in errors:
            print(f"    - {err}")

## Step 8: Verify Data and Create Summary Maps

Let's verify all the data looks correct and create some maps to visualize what we've downloaded.


In [ ]:
# SUMMARY: Print data summary statistics

print("\nData Summary:")
print("=" * 60)

print(f"\nStudy Area:")
if len(study_area_gdf) > 0:
    print(f"  CRS: {study_area_gdf.crs}")
    print(f"  Area: {study_area_gdf.geometry.area.values[0]:,.0f} sq feet")

print(f"\nParcels:")
if len(parcels) > 0:
    print(f"  Count: {len(parcels):,}")
    print(f"  CRS: {parcels.crs}")
    if 'parusedesc' in parcels.columns:
        print(f"  Land use breakdown (parusedesc):")
        for use, count in parcels['parusedesc'].value_counts().head(10).items():
            pct = 100 * count / len(parcels)
            print(f"    - {use}: {count:,} ({pct:.1f}%)")
        if parcels['parusedesc'].nunique() > 10:
            print(f"    ... and {parcels['parusedesc'].nunique() - 10} more categories")
else:
    print(f"  None downloaded")

print(f"\nFlood Zones:")
if len(flood_zones) > 0:
    print(f"  Count: {len(flood_zones):,}")
    print(f"  CRS: {flood_zones.crs}")
    if 'flood_category' in flood_zones.columns:
        print(f"  Zone breakdown:")
        for zone, count in flood_zones['flood_category'].value_counts().items():
            pct = 100 * count / len(flood_zones)
            print(f"    - {zone}: {count:,} ({pct:.1f}%)")
else:
    print(f"  None downloaded")

print(f"\nGeoPackage:")
print(f"  Path: {gpkg_filepath}")
if os.path.exists(gpkg_filepath):
    size_mb = os.path.getsize(gpkg_filepath) / (1024 * 1024)
    print(f"  Status: Saved to Google Drive ({size_mb:.2f} MB)")
    try:
        layers = fiona.listlayers(gpkg_filepath)
        print(f"  Layers: {', '.join(layers)}")
    except Exception:
        pass
else:
    print(f"  Status: FILE NOT FOUND — check cell above for errors")

print(f"\nData Folder Contents:")
for folder_name, folder_path in [('Base', BASE_DIR), ('Data', DATA_DIR), ('Outputs', OUTPUT_DIR)]:
    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        data_files = [f for f in files if not f.startswith('.')]
        print(f"  {folder_name} ({folder_path}):")
        if data_files:
            for f in sorted(data_files):
                fpath = os.path.join(folder_path, f)
                if os.path.isfile(fpath):
                    fsize = os.path.getsize(fpath) / 1024
                    print(f"    - {f} ({fsize:.1f} KB)")
                else:
                    print(f"    - {f}/ (folder)")
        else:
            print(f"    (empty)")
    else:
        print(f"  {folder_name}: folder does not exist")


### Static Map (Matplotlib)

Let's create a map showing all our data layers together.


In [ ]:
# MAP 1: Static map showing all layers

fig, ax = plt.subplots(figsize=(12, 8), facecolor='#2b2b2b')
ax.set_facecolor('#2b2b2b')

# Plot study area boundary
if len(study_area_gdf) > 0:
    study_area_gdf.plot(ax=ax, alpha=0.1, edgecolor='#00e5ff', linewidth=2, label='Study Area')

# Plot parcels
if len(parcels) > 0:
    parcels.plot(ax=ax, alpha=0.3, color='#555555', edgecolor='#666666', linewidth=0.5, label='Parcels')

# Plot flood zones
if len(flood_zones) > 0:
    colors = {
        'Floodway': '#2B5797',
        '100-year Flood (1%)': '#8FABBE',
        '500-year Flood (0.2%)': '#B4D4E7',
        'Other': '#B3E5FC'
    }

    for category, color in colors.items():
        subset = flood_zones[flood_zones['flood_category'] == category]
        if len(subset) > 0:
            subset.plot(ax=ax, alpha=0.6, color=color, edgecolor='#555555', linewidth=0.3, label=category)

ax.set_title('Swannanoa Watershed: All Data Layers', fontsize=16, fontweight='bold', color='white')
ax.legend(loc='upper right', fontsize=10, facecolor='#3a3a3a', edgecolor='#555555', labelcolor='white')
ax.legend_.get_title().set_color('white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#555555')
ax.grid(True, alpha=0.2, color='#555555')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'map_all_layers.png'), dpi=150, bbox_inches='tight',
            facecolor='#2b2b2b')
print("Saved: map_all_layers.png")
plt.show()

## Step 9: How to Use This Data in Desktop GIS

You've now created a GeoPackage file. Here's how to open it in QGIS or ArcGIS Pro.

### Using in QGIS (Free)

1. **Open QGIS** on your computer
2. **Click: Layer → Add Layer → Add Vector Layer**
3. **Select your GeoPackage file**:
   - Navigate to Google Drive → MSER_510_VULNERABILTY → outputs → Swannanoa_Study_Area.gpkg
4. **A dialog appears asking which layers to add** — select:
   - ☑ study_area
   - ☑ parcels
   - ☑ flood_zones
5. **Click Add** — all layers appear in your map!

### Using in ArcGIS Pro (Paid)

1. **Open ArcGIS Pro**
2. **In the Catalog pane (right side):**
   - Navigate to Databases
   - Right-click → Add Database Folder
   - Browse to your Google Drive outputs folder
3. **Drag each layer into your map**

### Next Steps

In the coming weeks, you'll analyze which properties are at risk from flooding, add elevation data, identify vulnerable populations, and create professional risk assessment maps.

---

## Summary

Congratulations! You've successfully:

- Installed and imported geographic libraries
- Connected to Google Drive for permanent storage
- Downloaded real parcel data from North Carolina
- Downloaded FEMA flood zone maps
- Downloaded building footprints from Overture Maps
- Standardized all data to the same coordinate system
- Created a GeoPackage file with 4 layers (study_area, parcels, flood_zones, buildings)
- Created maps to verify your data
- Learned how to use the data in QGIS and ArcGIS Pro

**Your GeoPackage is now ready for analysis!**

In the next class, we'll start analyzing which properties are at risk from flooding.

### Resources

- **QGIS**: https://www.qgis.org
- **ArcGIS Pro**: https://www.esri.com/en-us/arcgis/products/arcgis-pro
- **GeoPandas**: https://geopandas.org
- **NC OneMap**: https://www.nconemap.gov
- **FEMA Flood Maps**: https://msc.fema.gov
- **Overture Maps**: https://overturemaps.org